In [1]:
# !pip install jinja2

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations
from pathlib import Path
import pandas as pd
from jinja2 import Environment, FileSystemLoader, select_autoescape
import json
import datetime as dt

REPORTS_DIR = Path("data") / "reports"
CSV_SUM = REPORTS_DIR / "completude.csv"
TEMPLATES_DIR = REPORTS_DIR / "templates"
TEMPLATE_NAME = "completude.html.j2"
OUT_HTML = REPORTS_DIR / "completude.html"

def main():
    if not CSV_SUM.exists():
        raise FileNotFoundError(f"Não encontrei {CSV_SUM}. Gere o CSV de completude primeiro.")

    # Lê o CSV (sem suposições de ordem)
    df = pd.read_csv(CSV_SUM)
    needed = [
        "user", "assigned_total", "annotated_total", "pending", "percent_complete",
        "label_sim", "label_nao", "label_nao_sei", "extra_annotations", "last_update"
    ]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"{CSV_SUM.name} não tem a coluna obrigatória '{c}'. Colunas: {list(df.columns)}")

    # KPIs globais
    assigned_sum = int(df["assigned_total"].sum())
    annotated_sum = int(df["annotated_total"].sum())
    pct_global = (annotated_sum / max(1, assigned_sum)) * 100.0

    # Ordena por % completo desc, depois user
    df = df.sort_values(by=["percent_complete", "user"], ascending=[False, True]).reset_index(drop=True)

    # Dados para gráficos (Chart.js)
    chart_users = df["user"].astype(str).tolist()
    chart_pct = df["percent_complete"].astype(float).round(2).tolist()

    # Doughnut do global: parte concluída vs restante
    donut_data = {
        "labels": ["Concluído", "Restante"],
        "values": [round(pct_global, 2), round(100.0 - pct_global, 2)],
    }

    # Render com Jinja2
    env = Environment(
        loader=FileSystemLoader(str(TEMPLATES_DIR)),
        autoescape=select_autoescape(["html", "xml"]),
        trim_blocks=True,
        lstrip_blocks=True,
    )
    template = env.get_template(TEMPLATE_NAME)

    context = {
        "generated_at": dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "assigned_sum": f"{assigned_sum:,}",
        "annotated_sum": f"{annotated_sum:,}",
        "pct_global": f"{pct_global:.2f}",
        "rows": df.to_dict(orient="records"),

        # gráficos
        "chart_users_json": json.dumps(chart_users, ensure_ascii=False),
        "chart_pct_json": json.dumps(chart_pct, ensure_ascii=False),
        "donut_labels_json": json.dumps(donut_data["labels"], ensure_ascii=False),
        "donut_values_json": json.dumps(donut_data["values"], ensure_ascii=False),
    }

    html_out = template.render(**context)
    OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
    OUT_HTML.write_text(html_out, encoding="utf-8")
    print(f"✅ Relatório HTML gerado em: {OUT_HTML}")

In [3]:
main()

✅ Relatório HTML gerado em: data/reports/completude.html
